# Data Processing

This notebook maps the data preparation scripts into a notebook workflow for model-ready Arabica futures training data.

Outputs created here:
- `data/centralData/yahoo_cot_full_outer_by_date.csv`
- `data/centralData/yahoo_cot_full_outer_by_date_cot_ffill.csv`
- `data/centralData/arabica_ml_model_ready.csv`
- JSON/CSV audit files in `data/centralData/`

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent

CENTRAL_DATA_DIR = ROOT / "data" / "centralData"
CENTRAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable
print(f"Project root: {ROOT}")
print(f"Python: {PYTHON}")

## 1. Build Yahoo/COT Outer Join

This section preserves every Yahoo trading date and every Coffee C COT report date. COT is still sparse here; the next section forward-fills it in a bounded way for daily modeling.

In [ ]:
YAHOO_PRICE_FILE = ROOT / "data" / "yahoo" / "arabica_coffee_futures_history.csv"
COT_FEATURE_FILE = ROOT / "data" / "COT" / "coffee_c_all_cot_data.csv"

print(f"Yahoo input: {YAHOO_PRICE_FILE}")
print(f"COT input: {COT_FEATURE_FILE}")

In [ ]:
def load_yahoo_price_data(path=YAHOO_PRICE_FILE):
    yahoo = pd.read_csv(path)
    yahoo["Date"] = pd.to_datetime(yahoo["Date"], errors="coerce").dt.normalize()
    yahoo = yahoo.dropna(subset=["Date"]).copy()
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        yahoo[col] = pd.to_numeric(yahoo[col], errors="coerce")
    return yahoo.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")


def load_cot_feature_data(path=COT_FEATURE_FILE):
    cot = pd.read_csv(path, low_memory=False)
    cot_report_date = pd.to_datetime(cot["Report_Date_as_MM_DD_YYYY"], errors="coerce").dt.normalize()
    cot = pd.concat([cot, pd.DataFrame({"cot_report_date": cot_report_date})], axis=1)
    return cot.dropna(subset=["cot_report_date"]).copy()


yahoo_prices = load_yahoo_price_data()
cot_features = load_cot_feature_data()

print(f"Yahoo rows: {len(yahoo_prices):,}")
print(f"Yahoo date range: {yahoo_prices['Date'].min().date()} to {yahoo_prices['Date'].max().date()}")
print(f"COT rows: {len(cot_features):,}")
print(f"COT date range: {cot_features['cot_report_date'].min().date()} to {cot_features['cot_report_date'].max().date()}")

In [ ]:
def add_yahoo_derived_features(yahoo):
    yahoo = yahoo.sort_values("Date").copy()
    yahoo["return_1d"] = yahoo["Close"].pct_change()
    yahoo["return_5d"] = yahoo["Close"].pct_change(5)
    yahoo["range_pct"] = (yahoo["High"] - yahoo["Low"]) / yahoo["Close"]
    yahoo["close_to_open_pct"] = (yahoo["Close"] - yahoo["Open"]) / yahoo["Open"]
    yahoo["volume_change_pct"] = yahoo["Volume"].pct_change()
    yahoo["moving_avg_5"] = yahoo["Close"].rolling(5).mean()
    yahoo["moving_avg_20"] = yahoo["Close"].rolling(20).mean()
    yahoo["close_vs_ma_20"] = yahoo["Close"] / yahoo["moving_avg_20"] - 1
    yahoo["future_close_1d"] = yahoo["Close"].shift(-1)
    yahoo["future_close_5d"] = yahoo["Close"].shift(-5)
    yahoo["future_return_1d"] = yahoo["future_close_1d"] / yahoo["Close"] - 1
    yahoo["future_return_5d"] = yahoo["future_close_5d"] / yahoo["Close"] - 1
    yahoo["future_direction_5d"] = yahoo["future_return_5d"] > 0
    return yahoo


def add_cot_derived_features(cot):
    cot = cot.copy()
    numeric_prefixes = (
        "Open_Interest", "Prod_Merc", "Swap", "M_Money", "Other_Rept", "Tot_Rept",
        "NonRept", "Change_in", "Pct_of", "Traders", "Conc", "NonComm", "Comm",
    )
    for col in [c for c in cot.columns if c.startswith(numeric_prefixes)]:
        cot[col] = pd.to_numeric(cot[col], errors="coerce")

    formulas = {
        "managed_money_net": ("M_Money_Positions_Long_ALL", "M_Money_Positions_Short_ALL"),
        "managed_money_net_pct_oi": ("Pct_of_OI_M_Money_Long_All", "Pct_of_OI_M_Money_Short_All"),
        "producer_merchant_net": ("Prod_Merc_Positions_Long_ALL", "Prod_Merc_Positions_Short_ALL"),
        "commercial_net": ("Comm_Positions_Long_All", "Comm_Positions_Short_All"),
        "noncommercial_net": ("NonComm_Positions_Long_All", "NonComm_Positions_Short_All"),
        "nonreportable_net": ("NonRept_Positions_Long_All", "NonRept_Positions_Short_All"),
        "swap_dealer_net": ("Swap_Positions_Long_All", "Swap__Positions_Short_All"),
        "other_reportable_net": ("Other_Rept_Positions_Long_ALL", "Other_Rept_Positions_Short_ALL"),
        "managed_money_weekly_net_change": ("Change_in_M_Money_Long_All", "Change_in_M_Money_Short_All"),
        "commercial_weekly_net_change": ("Change_in_Comm_Long_All", "Change_in_Comm_Short_All"),
        "noncommercial_weekly_net_change": ("Change_in_NonComm_Long_All", "Change_in_NonComm_Short_All"),
    }
    derived_features = {}
    for new_col, (long_col, short_col) in formulas.items():
        if long_col in cot.columns and short_col in cot.columns:
            derived_features[new_col] = pd.to_numeric(cot[long_col], errors="coerce") - pd.to_numeric(cot[short_col], errors="coerce")
    if "Change_in_Open_Interest_All" in cot.columns and "Open_Interest_All" in cot.columns:
        derived_features["open_interest_change_pct"] = pd.to_numeric(cot["Change_in_Open_Interest_All"], errors="coerce") / pd.to_numeric(cot["Open_Interest_All"], errors="coerce")
    if derived_features:
        cot = pd.concat([cot, pd.DataFrame(derived_features, index=cot.index)], axis=1).copy()
    return cot


yahoo_features = add_yahoo_derived_features(yahoo_prices)
cot_features_enriched = add_cot_derived_features(cot_features)
print(f"Yahoo feature columns: {len(yahoo_features.columns):,}")
print(f"COT feature columns: {len(cot_features_enriched.columns):,}")

In [ ]:
def full_outer_join_yahoo_and_cot(yahoo, cot):
    merged = yahoo.merge(
        cot,
        how="outer",
        left_on="Date",
        right_on="cot_report_date",
        indicator=True,
        suffixes=("_yahoo", "_cot"),
    )
    merged["Date"] = merged["Date"].combine_first(merged["cot_report_date"])
    merged["date_match_status"] = merged["_merge"].map({"both": "both", "left_only": "yahoo_only", "right_only": "cot_only"})
    sort_columns = [col for col in ["Date", "source_dataset", "source_archive"] if col in merged.columns]
    merged = merged.sort_values(sort_columns, na_position="last").reset_index(drop=True).drop(columns=["_merge"])
    first_cols = [col for col in ["Date", "date_match_status", "cot_report_date"] if col in merged.columns]
    return merged[first_cols + [col for col in merged.columns if col not in first_cols]]


central_data = full_outer_join_yahoo_and_cot(yahoo_features, cot_features_enriched)
central_data_file = CENTRAL_DATA_DIR / "yahoo_cot_full_outer_by_date.csv"
central_data.to_csv(central_data_file, index=False)

join_summary = central_data["date_match_status"].value_counts(dropna=False).rename_axis("date_match_status").reset_index(name="rows")
join_summary_file = CENTRAL_DATA_DIR / "yahoo_cot_date_join_summary.csv"
join_summary.to_csv(join_summary_file, index=False)

unmatched_dates_file = CENTRAL_DATA_DIR / "yahoo_cot_unmatched_dates.csv"
unmatched_cols = [c for c in ["Date", "date_match_status", "source_dataset", "source_archive", "Market_and_Exchange_Names"] if c in central_data.columns]
central_data.loc[central_data["date_match_status"].isin(["yahoo_only", "cot_only"]), unmatched_cols].to_csv(unmatched_dates_file, index=False)

print(f"Wrote central joined data: {central_data_file}")
print(f"Wrote join summary: {join_summary_file}")
print(f"Wrote unmatched dates: {unmatched_dates_file}")
join_summary

## 2. Forward-Fill COT and Build Model-Ready Dataset

These cells map directly to the scripts created for ML cleanup. They keep all valid Yahoo OHLCV rows, forward-fill weekly COT values onto daily dates with a 10-day freshness limit, join local weather features, and drop non-core features with excessive missingness.

In [ ]:
def run_script(*args):
    cmd = [PYTHON, *args]
    print(" ".join(str(part) for part in cmd))
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

run_script("Scripts/project/scripts/fill_cot_forward_daily.py")
run_script("Scripts/project/scripts/prepare_ml_dataset.py")

## 3. Audit Model-Ready Data

In [ ]:
ready_path = CENTRAL_DATA_DIR / "arabica_ml_model_ready.csv"
report_path = CENTRAL_DATA_DIR / "arabica_ml_model_ready.report.json"
ready = pd.read_csv(ready_path, parse_dates=["Date"], low_memory=False)
report = json.loads(report_path.read_text())

core_columns = ["Date", "Close", "High", "Low", "Open", "Volume"]
summary = {
    "rows": len(ready),
    "columns": len(ready.columns),
    "unique_dates": ready["Date"].nunique(),
    "date_start": ready["Date"].min().date().isoformat(),
    "date_end": ready["Date"].max().date().isoformat(),
    "core_missing_cells": int(ready[core_columns].isna().sum().sum()),
    "weather_columns": len([c for c in ready.columns if c.startswith("weather_")]),
    "target_rows_5d": int(ready["target_return_5d"].notna().sum()) if "target_return_5d" in ready else None,
    "dropped_for_missing_count": report.get("dropped_for_missing_count"),
}
summary

In [ ]:
ready.head()